In [ ]:
import json
import re
import textwrap
import requests
import wikipedia
from docx import Document

# ==========================================
# 1. CONFIGURATION CONSTANTS
# ==========================================
# Default LM Studio local endpoint
LM_STUDIO_URL = "http://localhost:1234/v1/chat/completions"

# Replace this string with the exact model identifier currently loaded in your LM Studio UI
MODEL_NAME = "meta-llama-3-8b-instruct" 

# ==========================================
# 2. CORE WIKIPEDIA FUNCTION
# ==========================================
def search_wikipedia(query: str) -> str:
    """
    Searches Wikipedia for a given topic and returns a brief summary.
    """
    try:
        # Set a clear user agent to comply with Wikipedia's official API policy
        wikipedia.set_user_agent("LMStudioAgent/1.0 (contact: student_project@example.com)")
        
        # Fetch a concise 3-sentence summary of the topic
        summary = wikipedia.summary(query, sentences=3)
        return summary
    except wikipedia.exceptions.DisambiguationError as e:
        # If the term is too broad, return the first few alternative options
        return f"The term '{query}' is too ambiguous. Did you mean one of these? {', '.join(e.options[:5])}"
    except wikipedia.exceptions.PageError:
        return f"No Wikipedia page found matching '{query}'."
    except Exception as e:
        return f"An unexpected error occurred: {str(e)}"

# ==========================================
# 3. LLM TOOL DEFINITION (JSON SCHEMA)
# ==========================================
wikipedia_tool = {
    "type": "function",
    "function": {
        "name": "search_wikipedia",
        "description": "Search Wikipedia to retrieve factual, up-to-date summaries about people, places, events, technologies, or concepts.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The specific search term or precise Wikipedia article title to look up.",
                }
            },
            "required": ["query"],
        },
    },
}

# ==========================================
# 4. INTERACTIVE AGENT LOOP
# ==========================================
def run_agent():
    print("=== LM Studio Wikipedia Agent Console ===")
    print("Type your question below. (Type 'exit' or 'quit' to stop)\n")
    
    headers = {"Content-Type": "application/json"}
    
    while True:
        # Capture dynamic user input from the terminal
        user_prompt = input("\nYou: ").strip()
        
        # Break condition to close the application cleanly
        if user_prompt.lower() in ['exit', 'quit']:
            print("Exiting agent console. Goodbye!")
            break
            
        if not user_prompt:
            continue
            
        # Initialize a fresh message list for this specific question
        messages = [
            {"role": "user", "content": user_prompt}
        ]
        
        payload = {
            "model": MODEL_NAME,
            "messages": messages,
            "tools": [wikipedia_tool],
            "tool_choice": "auto",
            "temperature": 0.7
        }
        
        print("Thinking...")
        
        try:
            # Step 1: Send user prompt to LM Studio along with the available tool configuration
            response = requests.post(LM_STUDIO_URL, headers=headers, data=json.dumps(payload))
            if response.status_code != 200:
                print(f"Error from LM Studio: {response.status_code} - {response.text}")
                continue
                
            response_data = response.json()
            assistant_message = response_data['choices'][0]['message']
            
            # Step 2: Check if the model decided it needs to use the tool
            if 'tool_calls' in assistant_message and assistant_message['tool_calls']:
                tool_calls = assistant_message['tool_calls']
                
                # Append the model's tool call intent to the chat history
                messages.append(assistant_message)
                
                for tool_call in tool_calls:
                    if tool_call['function']['name'] == "search_wikipedia":
                        # Parse the search arguments generated by the model
                        args = json.loads(tool_call['function']['arguments'])
                        search_term = args.get("query")
                        
                        print(f"🤖 [LLM triggered Tool]: search_wikipedia(query='{search_term}')")
                        
                        # Run the local python scraping task
                        tool_result = search_wikipedia(search_term)
                        print(f"🌐 [Tool Result]: {tool_result}")
                        
                        # Append the tool's output back into the message history array
                        messages.append({
                            "role": "tool",
                            "tool_call_id": tool_call['id'],
                            "name": "search_wikipedia",
                            "content": tool_result
                        })
                
                # Step 3: Request the final synthesized answer compilation from the LLM
                print("Synthesizing final answer...")
                final_payload = {
                    "model": MODEL_NAME,
                    "messages": messages,
                    "temperature": 0.7
                }
                
                final_response = requests.post(LM_STUDIO_URL, headers=headers, data=json.dumps(final_payload))
                final_data = final_response.json()
                print(f"\nAI: {final_data['choices'][0]['message']['content']}")
                
            else:
                # If the model already knew the answer or chose not to use the tool
                print(f"\nAI: {assistant_message['content']}")
                
        except requests.exceptions.ConnectionError:
            print("\nConnection Error: Failed to connect to LM Studio.")
            print("Please ensure LM Studio is running and the Local Server toggle is set to ON at http://localhost:1234")
            break  # Exit the loop if the server connection drops

if __name__ == "__main__":
    run_agent()

=== LM Studio Wikipedia Agent Console ===
Type your question below. (Type 'exit' or 'quit' to stop)

Thinking...
🤖 [LLM triggered Tool]: search_wikipedia(query='agentic ai')
🌐 [Tool Result]: In the context of generative artificial intelligence, AI agents (also referred to as compound AI systems or agentic AI) are a class of intelligent agents that can pursue goals, use tools, and take actions with varying degrees of autonomy. In practice, they usually operate within human-defined objectives, constraints, and available tools.


== Overview ==
AI agents possess several key attributes, including goal-directed behavior, natural language interfaces, the capacity to use external tools, and the ability to perform multi-step tasks.
Synthesizing final answer...

AI: More specifically, Agentic AI refers to a type of artificial intelligence that is designed to act in the world, make decisions, and take actions based on its goals and objectives. It is inspired by the concept of agency, which refer